# 🚀 TNHComfy — Remote Worker on Google Colab

Run your own high-performance **TNHComfy Serverless Compute Worker** on Google Colab (Pro/Free).

### Supported Worker Modes:
- 🔮 **Hybrid Mode (Recommended)**: All-in-one node running **ComfyUI (Image/TTS)**, **Storage Agent**, and **llama.cpp (LLM)** concurrently on a single GPU. Supports automatic VRAM management (budget L4 cooperative yielding, A100-40GB semi-concurrency, A100-80GB parallel unconstrained).
- 🎨 **Image Worker (`remote_comfy`)**: Runs ComfyUI on `:8188` + Agent sidecar on `:8001` (with Google Drive checkpoint & LoRA caching).
- 💬 **LLM Worker (`remote_llm`)**: Runs CUDA-accelerated `llama-server` on `:8003` serving curated RP & Utility models from Google Drive.
- 🎙️ **TTS Worker (`remote_tts`)**: Runs Kokoro-82M / TTS model inference.

### Features:
- 🌐 **NetBird Mesh VPN**: Static private IP across reconnects. Setup key is only entered **once** and permanently backed up to Google Drive.
- 🔒 **Private Repository Access**: Works with private TNHComfy repository via GitHub PAT or Colab Secrets.
- 🔑 **Zero-Secret Re-entry**: HF_TOKEN and CIVITAI_TOKEN are handled by the backend.

In [ ]:
#@title 1. 📂 Mount Google Drive & 🔍 Check GPU Hardware
import os
import subprocess
import torch

# 1. Mount Google Drive for persistence (NetBird state, ComfyUI models, LLM weights)
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        print("📁 Mounting Google Drive...")
        drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully at /content/drive/MyDrive")
except Exception as e:
    print(f"⚠️ Note: Google Drive not mounted: {e}")

print("\n" + "=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"✅ CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select GPU.")
print("=" * 60 + "\n")

subprocess.run(["nvidia-smi"], check=False)


In [ ]:
#@title 2. ⚙️ Worker Configuration { run: "client" }
#@markdown Choose your worker mode and networking setup:

WORKER_MODE = "hybrid (all-in-one: image + llm + tts)" #@param ["hybrid (all-in-one: image + llm + tts)", "image (remote_comfy)", "llm (remote_llm)", "tts (remote_tts)"]
TUNNEL_METHOD = "netbird (private mesh VPN)" #@param ["netbird (private mesh VPN)", "cloudflared (free, no setup)"]

#@markdown ---
#@markdown #### ⚡ Concurrency & Hardware Policy (Hybrid Mode):
#@markdown - `auto`: Probes GPU VRAM automatically (>=70GB -> concurrent, >=35GB -> semi_concurrent, else exclusive)
#@markdown - `exclusive`: Mutual exclusion on GPU; ComfyUI cooperatively frees VRAM before LLM executes (Best for L4 24GB / T4)
#@markdown - `semi_concurrent`: Runs RP LLM and ComfyUI simultaneously; swaps utility models exclusively (Best for A100 40GB)
#@markdown - `concurrent`: Full parallel execution without restrictions (Best for A100 80GB / H100)
CONCURRENCY_POLICY = "auto (detect from GPU VRAM)" #@param ["auto (detect from GPU VRAM)", "exclusive (budget L4/T4)", "semi_concurrent (A100 40GB)", "concurrent (A100 80GB)"]

#@markdown ---
#@markdown #### 🌐 NetBird Mesh VPN (Default & Recommended):
#@markdown Your setup key is only needed ONCE. Config will be saved to Google Drive so your IP is static forever:
NETBIRD_SETUP_KEY = "" #@param {type:"string"}

#@markdown ---
#@markdown #### 🔒 Private Repository Access (TNHComfy is private):
#@markdown Enter a GitHub Personal Access Token (PAT) with repo read permissions (or save as GITHUB_TOKEN in Colab Secrets):
GITHUB_TOKEN = "" #@param {type:"string"}

#@markdown ---
#@markdown #### 🎨 Image / Hybrid Mode — Google Drive ComfyUI:
#@markdown Path to your ComfyUI folder copied from Modal (default is /content/drive/MyDrive/comfyui):
GDRIVE_COMFY_PATH = "/content/drive/MyDrive/comfyui" #@param {type:"string"}

#@markdown ---
#@markdown #### 💬 LLM / Hybrid Mode — Curated Models from models.json:
LLM_RP_MODEL = "artemis_31b" #@param ["artemis_31b", "writers_31b_v2", "gemma4_31b_heretic", "sompoa_26b_a4b", "weirdcompound_24b", "magnum_diamond_24b"]
LLM_UTIL_MODEL = "same_as_rp" #@param ["same_as_rp", "gemma3_12b_heretic_util", "mistral_small_32_it"]
GDRIVE_LLM_PATH = "/content/drive/MyDrive/tnhcomfy/llm_models" #@param {type:"string"}

# Try resolving tokens from Google Colab Secrets (the Key icon on left sidebar)
try:
    from google.colab import userdata
    if not GITHUB_TOKEN: GITHUB_TOKEN = userdata.get("GITHUB_TOKEN", "")
    if not NETBIRD_SETUP_KEY: NETBIRD_SETUP_KEY = userdata.get("NETBIRD_SETUP_KEY", "")
except Exception:
    pass

print("⚙️ Configuration saved!")
print(f"• Mode: {WORKER_MODE}")
print(f"• Network: {TUNNEL_METHOD}")
if "hybrid" in WORKER_MODE:
    print(f"• Policy: {CONCURRENCY_POLICY}")
    print(f"• ComfyUI Drive Path: {GDRIVE_COMFY_PATH}")
    print(f"• LLM Drive Path: {GDRIVE_LLM_PATH}")
elif "image" in WORKER_MODE:
    print(f"• ComfyUI Drive Path: {GDRIVE_COMFY_PATH}")
elif "llm" in WORKER_MODE:
    print(f"• LLM RP Model: {LLM_RP_MODEL}")
    print(f"• LLM Util Model: {LLM_UTIL_MODEL}")


In [ ]:
#@title 3. 📥 Clone / Update TNHComfy Repository
import os
import subprocess

repo_dir = "/content/tnhcomfy"
token = GITHUB_TOKEN.strip()

if token:
    clone_url = f"https://{token}@github.com/TheNaha/TNHComfy.git"
else:
    clone_url = "https://github.com/TheNaha/TNHComfy.git"

if not os.path.exists(repo_dir):
    print("📥 Cloning TNHComfy repository...")
    subprocess.run(["git", "clone", clone_url, repo_dir], check=True)
else:
    print("🔄 Updating existing repository...")
    subprocess.run(["git", "-C", repo_dir, "remote", "set-url", "origin", clone_url], check=True)
    subprocess.run(["git", "-C", repo_dir, "pull"], check=True)

print("✅ TNHComfy workspace ready at:", repo_dir)


In [ ]:
#@title 4. 📦 Install Dependencies & Prepare Environment
import os
import sys
import shutil
import urllib.request
import subprocess

mode_key = "hybrid" if "hybrid" in WORKER_MODE else ("model" if "image" in WORKER_MODE else ("llm" if "llm" in WORKER_MODE else "tts"))
tunnel_key = "netbird" if "netbird" in TUNNEL_METHOD else "cloudflared"

print(f"📦 Setting up environment for {WORKER_MODE} via {TUNNEL_METHOD}...")

# 1. Install remote_agent requirements
agent_dir = "/content/tnhcomfy/remote_agent"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{agent_dir}/requirements.txt"], check=True)

# 2. Install Networking Tool
if tunnel_key == "netbird":
    if not shutil.which("netbird"):
        print("📥 Installing NetBird client...")
        subprocess.run("curl -fsSL https://pkgs.netbird.io/install.sh | sh", shell=True, check=True)
    print("✅ NetBird installed.")
else:
    if not shutil.which("cloudflared"):
        print("📥 Downloading Cloudflared binary...")
        cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        urllib.request.urlretrieve(cf_url, "/usr/local/bin/cloudflared")
        os.chmod("/usr/local/bin/cloudflared", 0o755)
    print("✅ Cloudflared installed.")

# 3. ComfyUI Setup (if Image or Hybrid mode)
if mode_key in ("model", "hybrid"):
    comfy_dir = "/content/ComfyUI"
    gdrive_path = GDRIVE_COMFY_PATH.strip()
    
    # Check if user has entire ComfyUI or models in Google Drive
    if os.path.exists(gdrive_path):
        print(f"🎨 Found existing ComfyUI directory in Google Drive: {gdrive_path}")
        if os.path.exists(os.path.join(gdrive_path, "main.py")):
            if not os.path.exists(comfy_dir):
                os.symlink(gdrive_path, comfy_dir)
            print("🔗 Linked Google Drive ComfyUI to /content/ComfyUI")
        elif os.path.exists(os.path.join(gdrive_path, "models")):
            if not os.path.exists(comfy_dir):
                print("📥 Cloning ComfyUI core...")
                subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", comfy_dir], check=True)
            if os.path.exists(f"{comfy_dir}/models") and not os.path.islink(f"{comfy_dir}/models"):
                shutil.rmtree(f"{comfy_dir}/models")
            os.symlink(os.path.join(gdrive_path, "models"), f"{comfy_dir}/models")
            print("🔗 Linked Google Drive models directory!")
    else:
        if not os.path.exists(comfy_dir):
            print("📥 Cloning fresh ComfyUI core...")
            subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", comfy_dir], check=True)
    
    # Install ComfyUI requirements
    print("📦 Ensuring ComfyUI dependencies...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{comfy_dir}/requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate", "scipy"], check=True)
    print("✅ ComfyUI environment is ready.")

# 4. llama-server Setup (if LLM or Hybrid mode)
if mode_key in ("llm", "hybrid"):
    if not shutil.which("llama-server") and not os.path.exists("/usr/local/bin/llama-server"):
        print("📥 Fetching CUDA-accelerated llama-server binary...")
        llama_tar = "/tmp/llama_server.tar.gz"
        llama_url = "https://github.com/ggml-org/llama.cpp/releases/download/b11647/llama-b11647-bin-ubuntu-cuda-12.8-x64.tar.gz"
        urllib.request.urlretrieve(llama_url, llama_tar)
        os.makedirs("/tmp/llama_bin", exist_ok=True)
        subprocess.run(f"tar -xzf {llama_tar} -C /tmp/llama_bin", shell=True, check=True)
        for root, dirs, files in os.walk("/tmp/llama_bin"):
            if "llama-server" in files:
                src = os.path.join(root, "llama-server")
                shutil.copy(src, "/usr/local/bin/llama-server")
                os.chmod("/usr/local/bin/llama-server", 0o755)
                break
        print("✅ Installed CUDA-enabled llama-server to /usr/local/bin/llama-server")
    
    llm_dir = GDRIVE_LLM_PATH.strip() or "/content/drive/MyDrive/tnhcomfy/llm_models"
    os.makedirs(llm_dir, exist_ok=True)
    os.makedirs("/content/agent-data/llm", exist_ok=True)
    if not os.path.exists("/content/agent-data/llm/models"):
        os.symlink(llm_dir, "/content/agent-data/llm/models")
    print(f"✅ LLM storage ready at {llm_dir}")

print("\n🎉 Environment setup complete!")


In [ ]:
#@title 5. 🚀 Launch Worker & Start Networking
import os
import sys
import time
import re
import secrets
import subprocess
from IPython.display import display, HTML

mode_key = "hybrid" if "hybrid" in WORKER_MODE else ("model" if "image" in WORKER_MODE else ("llm" if "llm" in WORKER_MODE else "tts"))
tunnel_key = "netbird" if "netbird" in TUNNEL_METHOD else "cloudflared"
policy_key = "auto" if "auto" in CONCURRENCY_POLICY else ("exclusive" if "exclusive" in CONCURRENCY_POLICY else ("semi_concurrent" if "semi_concurrent" in CONCURRENCY_POLICY else "concurrent"))
gdrive_root = "/content/drive/MyDrive/tnhcomfy"
os.makedirs(gdrive_root, exist_ok=True)

# 1. Persistent Agent Service Token
token_file = os.path.join(gdrive_root, ".agent-token")
if os.path.exists(token_file):
    with open(token_file, "r") as f:
        agent_token = f.read().strip()
else:
    agent_token = secrets.token_urlsafe(32)
    with open(token_file, "w") as f:
        f.write(agent_token)
os.environ["REMOTE_AGENT_TOKEN"] = agent_token

agent_dir = "/content/tnhcomfy/remote_agent"
log_comfy = "/content/comfyui.log"
log_agent = "/content/agent.log"
log_llm = "/content/llama_server.log"
comfy_port = 8188
agent_port = 8001 if mode_key in ("model", "hybrid") else (8003 if mode_key == "llm" else 8002)
llm_port = 8003

# 2. Launch Background Processes
if mode_key == "hybrid":
    print("🚀 [Hybrid] Starting ComfyUI server on :8188...")
    comfy_dir = "/content/ComfyUI"
    with open(log_comfy, "w") as out:
        subprocess.Popen([sys.executable, f"{comfy_dir}/main.py", "--listen", "127.0.0.1", "--port", str(comfy_port)], stdout=out, stderr=out)
    
    print(f"🚀 [Hybrid] Starting Remote Storage Agent on :{agent_port} (DATA_DIR={comfy_dir})...")
    env_agent = dict(os.environ, DATA_DIR=comfy_dir, WORKER_MODE="hybrid", PORT=str(agent_port), HOST="127.0.0.1")
    with open(log_agent, "w") as out:
        subprocess.Popen([sys.executable, f"{agent_dir}/server.py"], cwd=agent_dir, env=env_agent, stdout=out, stderr=out)
    
    print(f"🚀 [Hybrid] Starting CUDA llama-server on :{llm_port}...")
    llm_dir = "/content/agent-data/llm/models"
    # Check if any .gguf exists to load initially
    initial_model = ""
    for root, dirs, files in os.walk(llm_dir):
        for f in files:
            if f.endswith(".gguf"):
                initial_model = os.path.join(root, f)
                break
        if initial_model: break
    
    llama_args = ["llama-server", "--port", str(llm_port), "--host", "127.0.0.1", "-ngl", "99"]
    if initial_model:
        llama_args.extend(["-m", initial_model])
    with open(log_llm, "w") as out:
        subprocess.Popen(llama_args, stdout=out, stderr=out)

elif mode_key == "model":
    print("🚀 Starting ComfyUI server on :8188...")
    comfy_dir = "/content/ComfyUI"
    with open(log_comfy, "w") as out:
        subprocess.Popen([sys.executable, f"{comfy_dir}/main.py", "--listen", "127.0.0.1", "--port", str(comfy_port)], stdout=out, stderr=out)
    
    print(f"🚀 Starting Remote Agent on :{agent_port} (DATA_DIR={comfy_dir})...")
    env = dict(os.environ, DATA_DIR=comfy_dir, WORKER_MODE="model", PORT=str(agent_port), HOST="127.0.0.1")
    with open(log_agent, "w") as out:
        subprocess.Popen([sys.executable, f"{agent_dir}/server.py"], cwd=agent_dir, env=env, stdout=out, stderr=out)

elif mode_key == "llm":
    llm_dir = "/content/agent-data/llm/models"
    print(f"🚀 Starting Remote LLM Agent on :{agent_port}...")
    env = dict(os.environ, DATA_DIR=llm_dir, LLM_DATA_DIR=llm_dir, WORKER_MODE="llm", PORT=str(agent_port), HOST="127.0.0.1")
    with open(log_agent, "w") as out:
        subprocess.Popen([sys.executable, f"{agent_dir}/server.py"], cwd=agent_dir, env=env, stdout=out, stderr=out)

elif mode_key == "tts":
    tts_dir = "/content/agent-data/tts"
    os.makedirs(tts_dir, exist_ok=True)
    print(f"🚀 Starting Remote TTS Agent on :{agent_port}...")
    env = dict(os.environ, TTS_DATA_DIR=tts_dir, WORKER_MODE="tts", PORT=str(agent_port), HOST="127.0.0.1")
    with open(log_agent, "w") as out:
        subprocess.Popen([sys.executable, f"{agent_dir}/server.py"], cwd=agent_dir, env=env, stdout=out, stderr=out)

# Wait for agent healthcheck
print("⏳ Waiting for agent healthcheck...")
for _ in range(40):
    try:
        import urllib.request
        with urllib.request.urlopen(f"http://127.0.0.1:{agent_port}/health", timeout=1) as res:
            if res.status == 200:
                print("✅ Agent is online and healthy!")
                break
    except Exception:
        time.sleep(1)

# 3. Establish Network Connection (NetBird with Persistence or Cloudflared)
base_url = ""
comfy_url = ""
agent_url = ""
llm_url = ""

if tunnel_key == "netbird":
    nb_persist = os.path.join(gdrive_root, "netbird")
    os.makedirs(nb_persist, exist_ok=True)
    subprocess.run("netbird service start || true", shell=True)
    
    if os.path.exists(os.path.join(nb_persist, "config.json")):
        print("🔄 Restoring existing NetBird peer credentials from Google Drive (Static IP preserved!)...")
        os.makedirs("/var/lib/netbird", exist_ok=True)
        subprocess.run(f"cp -rf {nb_persist}/* /var/lib/netbird/", shell=True, check=True)
        subprocess.run("netbird up", shell=True, check=True)
    else:
        print("🌐 Connecting NetBird for the first time...")
        nb_key = NETBIRD_SETUP_KEY.strip()
        if not nb_key:
            raise ValueError("NETBIRD_SETUP_KEY is required for first-time NetBird setup! Add it to the form or Colab Secrets.")
        subprocess.run(f"netbird up --setup-key {nb_key} --hostname comfy-colab-worker", shell=True, check=True)
        time.sleep(2)
        subprocess.run(f"cp -rf /var/lib/netbird/* {nb_persist}/", shell=True, check=True)
        print(f"💾 NetBird credentials permanently backed up to Google Drive ({nb_persist})!")
    
    time.sleep(2)
    nb_ip = subprocess.check_output("netbird status | grep -oE 'NetBird IP: [0-9.]+' | head -n1 | cut -d' ' -f3", shell=True, text=True).strip()
    print(f"✅ Connected to NetBird Mesh with Static IP: {nb_ip}")
    base_url = f"http://{nb_ip}"
    comfy_url = f"http://{nb_ip}:{comfy_port}"
    agent_url = f"http://{nb_ip}:{agent_port}"
    llm_url = f"http://{nb_ip}:{llm_port}"

else:
    print("🌐 Establishing Cloudflare public tunnels...")
    def start_cf_tunnel(port):
        log_path = f"/content/tunnel_{port}.log"
        subprocess.Popen(["cloudflared", "--no-autoupdate", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
        for _ in range(40):
            if os.path.exists(log_path):
                with open(log_path, "r") as f:
                    matches = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
                    if matches: return matches[0]
            time.sleep(1)
        return f"http://127.0.0.1:{port}"
    
    if mode_key == "hybrid":
        comfy_url = start_cf_tunnel(comfy_port)
        agent_url = start_cf_tunnel(agent_port)
        llm_url = start_cf_tunnel(llm_port)
        base_url = agent_url
    elif mode_key == "model":
        comfy_url = start_cf_tunnel(comfy_port)
        agent_url = start_cf_tunnel(agent_port)
    else:
        agent_url = start_cf_tunnel(agent_port)

# 4. Display Connection Card
if mode_key == "hybrid":
    provider_category = "Hybrid"
    provider_type = "remote_hybrid"
elif mode_key == "model":
    provider_category = "Image"
    provider_type = "remote_comfy"
elif mode_key == "llm":
    provider_category = "LLM"
    provider_type = "remote_llm"
else:
    provider_category = "TTS"
    provider_type = "remote_tts"

card_html = f"""
<div style="background: linear-gradient(135deg, #0f172a 0%, #1e1b4b 100%); border: 1px solid #3b82f6; border-radius: 16px; padding: 24px; color: #f8fafc; font-family: system-ui, -apple-system, sans-serif; box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.5); margin: 20px 0;">
    <h2 style="margin: 0 0 12px 0; color: #60a5fa; font-size: 20px; display: flex; align-items: center; gap: 8px;">
        ✨ TNHComfy Colab Worker is Ready! ({provider_category} Mode)
    </h2>
    <p style="margin: 0 0 16px 0; font-size: 14px; color: #94a3b8;">
        Your worker is online and connected. Copy the credentials below into your TNHComfy Providers page:
    </p>

    <div style="background: rgba(15, 23, 42, 0.85); border: 1px solid #334155; border-radius: 10px; padding: 16px; margin-bottom: 16px;">
        <div style="margin-bottom: 12px;">
            <span style="color: #64748b; font-size: 11px; text-transform: uppercase; font-weight: bold; letter-spacing: 0.05em;">Provider Category & Type</span>
            <div style="font-family: monospace; font-size: 14px; color: #38bdf8; font-weight: bold; margin-top: 2px;">{provider_category} &rarr; {provider_type}</div>
        </div>
        {"<div style='margin-bottom: 12px;'><span style='color: #64748b; font-size: 11px; text-transform: uppercase; font-weight: bold;'>Unified Base URL (base_url)</span><div style='font-family: monospace; font-size: 14px; color: #a5f3fc; word-break: break-all; margin-top: 2px;'>" + base_url + "</div></div>" if base_url else ""}
        {"<div style='margin-bottom: 12px;'><span style='color: #64748b; font-size: 11px; text-transform: uppercase; font-weight: bold;'>ComfyUI Server URL (comfy_url)</span><div style='font-family: monospace; font-size: 14px; color: #a5f3fc; word-break: break-all; margin-top: 2px;'>" + comfy_url + "</div></div>" if comfy_url else ""}
        <div style="margin-bottom: 12px;">
            <span style="color: #64748b; font-size: 11px; text-transform: uppercase; font-weight: bold;">Remote Agent URL (agent_url)</span>
            <div style="font-family: monospace; font-size: 14px; color: #a5f3fc; word-break: break-all; margin-top: 2px;">{agent_url}</div>
        </div>
        {"<div style='margin-bottom: 12px;'><span style='color: #64748b; font-size: 11px; text-transform: uppercase; font-weight: bold;'>LLM Server URL (llm_url)</span><div style='font-family: monospace; font-size: 14px; color: #a5f3fc; word-break: break-all; margin-top: 2px;'>" + llm_url + "</div></div>" if llm_url and mode_key == "hybrid" else ""}
        {"<div style='margin-bottom: 12px;'><span style='color: #64748b; font-size: 11px; text-transform: uppercase; font-weight: bold;'>Hardware Concurrency Policy</span><div style='font-family: monospace; font-size: 14px; color: #f59e0b; margin-top: 2px;'>" + policy_key + "</div></div>" if mode_key == "hybrid" else ""}
        <div>
            <span style="color: #64748b; font-size: 11px; text-transform: uppercase; font-weight: bold;">Remote Agent Token (agent_token)</span>
            <div style="font-family: monospace; font-size: 14px; color: #34d399; word-break: break-all; margin-top: 2px;">{agent_token}</div>
        </div>
    </div>

    <div style="font-size: 13px; color: #cbd5e1; background: rgba(59, 130, 246, 0.15); border-left: 4px solid #3b82f6; padding: 12px 16px; border-radius: 4px; line-height: 1.5;">
        💡 <b>Static Configuration:</b><br/>
        Because your NetBird state and token are saved in Google Drive, the IP and token above are <b>permanent</b>. You do not need to update your provider row when restarting Colab!
    </div>
</div>
"""
display(HTML(card_html))

print("\n📡 Keeping worker alive... Keep this cell running while using TNHComfy!")
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print("🛑 Worker stopped.")


In [ ]:
#@title 6. 📥 (Optional) Live Model Downloader & Manager
#@markdown Download models into your Google Drive models repository on the fly:

DOWNLOAD_TARGET = "ComfyUI Checkpoint" #@param ["ComfyUI Checkpoint", "ComfyUI LoRA", "ComfyUI VAE", "LLM GGUF Model"]
DOWNLOAD_URL = "" #@param {type:"string"}
FILENAME_OVERRIDE = "" #@param {type:"string"}

if DOWNLOAD_URL.strip():
    if "LLM" in DOWNLOAD_TARGET:
        dest_dir = GDRIVE_LLM_PATH.strip() or "/content/drive/MyDrive/tnhcomfy/llm_models"
    elif "Checkpoint" in DOWNLOAD_TARGET:
        dest_dir = "/content/ComfyUI/models/checkpoints"
    elif "LoRA" in DOWNLOAD_TARGET:
        dest_dir = "/content/ComfyUI/models/loras"
    else:
        dest_dir = "/content/ComfyUI/models/vae"
    
    os.makedirs(dest_dir, exist_ok=True)
    fname = FILENAME_OVERRIDE.strip() or DOWNLOAD_URL.split("/")[-1].split("?")[0]
    dest_path = os.path.join(dest_dir, fname)
    print(f"📥 Downloading {fname} to {dest_path}...")
    subprocess.run(f"curl -fL '{DOWNLOAD_URL.strip()}' -o '{dest_path}'", shell=True, check=True)
    print(f"✅ Successfully downloaded to {dest_path}!")
else:
    print("Please enter a valid DOWNLOAD_URL.")


In [ ]:
#@title 7. ⏱️ Prevent Colab Disconnect (Keep-Alive Audio Context)
from IPython.display import Audio, display
import numpy as np

# Generates an inaudible tone to keep Google Colab audio context awake
framerate = 44100
t = np.linspace(0, 5, framerate * 5)
data = np.sin(2 * np.pi * 20 * t) * 0.001
display(Audio(data, rate=framerate, autoplay=True))
print("🔊 Background keep-alive initialized.")
